# Displacement

**Project:** Pali Wacth

**Notebook:** `displacement.ipynb`

**Author:** Yuseof J

**Created:** 18/11/2026 | **Last updated:** 18/11/2026

## Purpose
   This file analyzes displacement pressures by governorate

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
    - Data runs from Jan 2025 to Oct 2026, so historical analysis is limited accordingly

In [ ]:
import datetime
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

# Data imports / exports

In [ ]:
# import paths
PATH_TO_DISPLACEMENT_EVENTS = '../data/raw/pse_idmc_idu_events.csv'
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = '../data/processed/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = '../data/processed/acled/conflict_events_geolocated.csv'

# import data
df_displacement = pd.read_csv(PATH_TO_DISPLACEMENT_EVENTS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

# Preprocessing

In [ ]:
date_col_to_use = 'displacement_end_date'

# format date columns
df_displacement[date_col_to_use] = pd.to_datetime(df_displacement[date_col_to_use])

# create date grouping columns
df_displacement['YEAR'] = df_displacement[date_col_to_use].dt.year
df_displacement['YEARMO'] = df_displacement[date_col_to_use].dt.to_period('M').astype(str)
df_displacement['YEAR_WEEK'] = df_displacement[date_col_to_use].dt.to_period('W').astype(str)

# EDA

In [ ]:
# df_displacement.columns.tolist()

In [ ]:
df_displacement.head()

## Geographic Distribution

In [ ]:
print(f"{df_displacement.centroid.nunique()} unique centroids")
print(f"{df_displacement.latitude.nunique()} unique latitudes")
print(f"{df_displacement.longitude.nunique()} unique longitudes")
print(f"{df_displacement.locations_type.nunique()} unique locations_type")
print(f"{df_displacement.locations_name.nunique()} unique locations_name")

### TODO: rich info in locations_name, can be distilled for finer geographic granularity

In [ ]:
# df_displacement.locations_name.tolist()

## Temporal Distribution

### TODO: date columns don't always align with 'description' column (which contains dates of displacement

In [ ]:
print(f"Earliest start date: {df_displacement.displacement_start_date.min()}")
print(f"Latest start date: {df_displacement.displacement_start_date.max()}")
print(f"Earliest end date: {df_displacement.displacement_end_date.min()}")
print(f"Latest end date: {df_displacement.displacement_end_date.max()}")

# Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_displacement = gpd.GeoDataFrame(
    df_displacement,
    geometry=gpd.points_from_xy(
        df_displacement['longitude'],
        df_displacement['latitude']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_displacement.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_displacement = gdf_displacement.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_displacement.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_displacement_joined = gpd.sjoin(
    gdf_displacement,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_displacement_joined))
print("Matched events:", gdf_displacement_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_displacement_joined['adm1_name'].isna().sum())

### TODO: handle unmatched events

In [ ]:
unmatched = gdf_events_joined[gdf_events_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

# Q1: Where is displacement happening (overall/across time)?

**While the West Bank has seen a greater amount of displacement EVENTS, Gaza experienced a far greater amount of displaced INDIVIDUALS**

## TODO: Normalize event counts by population since comparing regions of different sizes.

### Total num displacement events per admin 

In [ ]:
 gdf_displacement_joined['adm1_name'].value_counts()

In [ ]:
# gdf_displacement_joined.groupby(['adm1_name']).adm2_name.value_counts()
result = (
    gdf_displacement_joined
    .groupby('adm1_name')['adm2_name']
    .value_counts()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x.sort_values(ascending=False))
)
result

### Total num displaced people per admin

In [ ]:
gdf_displacement_joined.groupby('adm1_name').figure.sum()

In [ ]:
#gdf_displacement_joined.groupby(['adm1_name', 'adm2_name']).figure.sum()
result = (
    gdf_displacement_joined
    .groupby(['adm1_name', 'adm2_name'])['figure']
    .sum()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x.sort_values(ascending=False))
)
result

# Q2: Where is displacement changing the fastest?

The mothodology here is as follows:

A trend line is fit to each governorate. Two versions: num of displacement incidents, and num displaced people. 
The trend will indicate the pace at which a situation is deteriorating.

For sudden spikes/surges in displacement, residuals are used to flag any observations that have significantly larger num events/people than the trend would have otherwise expected

In [ ]:
gdf_displacement.head()

NOTE: treating the displacement end date as the date that everyone from this row was displaced, even though they were likely displaced over the period from start to finish date

In [ ]:
df_displacement[['event_end_date', 'YEAR', 'YEARMO', 'YEAR_WEEK']]

In [ ]:
make sure to flag surges (outlier detection presumably)
make sure to flag first time incidents (within x radius of that specific geography)

Normalize event counts by population if comparing regions of substantially different sizes.

Linear Regression

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def plot_displacement_trends(
    df,
    value_col,
    date_col,
    separate=False,
    cumulative=False,
    show_values=True
):

    plot_df = df.copy()

    # Format 'YEAR_WEEK' so that it works with the funtctions below 
    # e.g. the week range "2026-09-21/2026-09-27" has to be a single date
    # this solution will take the end date of the range    if date_granularity == 'YEAR_WEEK':
    plot_df[date_col] = pd.to_datetime(
        plot_df[date_col].str.split('/').str[1]
    )

    plot_df[date_col] = pd.to_datetime(plot_df[date_col])

    # Sort chronologically
    plot_df = plot_df.sort_values(date_col)

    # Calculate cumulative values independently per geography
    plot_col = value_col

    if cumulative:
        plot_col = f'{value_col}_cumulative'
        plot_df[plot_col] = (
            plot_df.groupby('adm2_name')[value_col].cumsum()
        )

    if separate:
        geographies = plot_df['adm2_name'].unique()

        fig, axes = plt.subplots(
            len(geographies), 1,
            figsize=(14, 5 * len(geographies)),
            sharex=True
        )

        if len(geographies) == 1:
            axes = [axes]

        for geo, ax in zip(geographies, axes):
            subset = plot_df[plot_df['adm2_name'] == geo]

            sns.lineplot(
                data=subset,
                x=date_col,
                y=plot_col,
                marker='o',
                ax=ax
            )

            # Display values at each point
            if show_values:
                for _, row in subset.iterrows():
                    ax.annotate(
                        f'{row[plot_col]:,.0f}',
                        (row[date_col], row[plot_col]),
                        textcoords='offset points',
                        xytext=(0, 8),
                        ha='center',
                        fontsize=8
                    )

            ax.set_title(geo)
            ax.set_ylabel(plot_col)

        plt.xlabel('Date')

    else:
        fig, ax = plt.subplots(figsize=(14, 6))

        sns.lineplot(
            data=plot_df,
            x=date_col,
            y=plot_col,
            hue='adm2_name',
            marker='o',
            ax=ax
        )

        # Display values at each point
        if show_values:
            for _, row in plot_df.iterrows():
                ax.annotate(
                    f'{row[plot_col]:,.0f}',
                    (row[date_col], row[plot_col]),
                    textcoords='offset points',
                    xytext=(0, 8),
                    ha='center',
                    fontsize=8
                )

        ax.set_title(f'{plot_col} Over Time by Geography')
        ax.set_xlabel('Date')
        ax.set_ylabel(plot_col)
        ax.legend(title='Geography')

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
#date_granularity = 'YEARMO' 
date_granularity = 'YEAR_WEEK'
regression_group = ['adm2_name', date_granularity]

# get data in proper format: region, yearmo-week
gdf_displacement_rate = gdf_displacement_joined.groupby(regression_group).agg(
    num_events=('figure', 'size'),
    num_displaced=('figure', 'sum')
)

In [ ]:
# filter out gaza (displacement figures dwarf WB values)
wb_admins = gdf_displacement_joined[gdf_displacement_joined.adm1_name=='West Bank'].adm2_name.unique()
df = gdf_displacement_rate.reset_index().copy()
wb_filter = df.adm2_name.isin(wb_admins)
df_wb = df[wb_filter]

In [ ]:
df_wb

In [ ]:
# plot_displacement_trends(df_wb, 'num_displaced', date_granularity, separate=True)

### Cumulative displacement by governorate

In [ ]:
plot_displacement_trends(df[wb_filter], 'num_displaced', date_granularity, separate=True, cumulative=True)

### Days since last displacement

Some govornorates experience a dropoff in displacement events, so its important to know how long it's been since last displacement

In [ ]:
# get the most recent displacement event per gov
gdf_displacement_recency = gdf_displacement_joined.groupby('adm2_name').agg(
    last_displacement=('displacement_end_date', 'max'),
).reset_index()

# subtract from today's date
today = datetime.today()
gdf_displacement_recency['days_since_last_displacement_event'] = gdf_displacement_recency.last_displacement.apply(lambda x: x

# how many days since last displacement event 
gdf_displacement_recency.sort_values('last_displacement', ascending=False)

Number of displacement events in last X days

Number displaced in last X days